# UdaPlay — Part 2: Agent Implementation

**Project:** UdaPlay – AI Research Agent for the Video Game Industry  
**Notebook:** `Udaplay_02_solution_project.ipynb`

This notebook builds the full UdaPlay agent:
1. **Three core tools** — `retrieve_game`, `evaluate_retrieval`, `game_web_search`
2. **State machine workflow** — RETRIEVE → EVALUATE → (ANSWER | WEB_SEARCH) → REPORT
3. **Stateful agent** — maintains conversation history across multiple queries
4. **Demonstration** — runs the agent on 5 example queries with full reasoning trace

## 0. Environment Setup

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv("config.env")

assert os.getenv("OPENAI_API_KEY") is not None
assert os.getenv("TAVILY_API_KEY") is not None

OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL", "https://openai.vocareum.com/v1")

print("✅ Environment loaded")

In [ ]:
import json
import re
from enum import Enum, auto
from dataclasses import dataclass, field
from typing import Optional

import chromadb
from chromadb.utils import embedding_functions
from openai import OpenAI
from pydantic import BaseModel
from tavily import TavilyClient

print("✅ Libraries imported")

## 1. Initialise Clients

In [ ]:
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=OPENAI_BASE_URL
)

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

# Reconnect to the ChromaDB collection created in Part 1
CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "udaplay_games"

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.getenv("OPENAI_API_KEY"),
    api_base=OPENAI_BASE_URL,
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)
collection = chroma_client.get_collection(
    name=COLLECTION_NAME,
    embedding_function=openai_ef
)

print(f"✅ All clients initialised")
print(f"   ChromaDB collection: {COLLECTION_NAME} ({collection.count()} documents)")

## 2. Structured Output Models (Pydantic)

In [ ]:
class RetrievalEvaluation(BaseModel):
    """Structured output from the evaluate_retrieval tool."""
    confidence: float          # 0.0 – 1.0
    is_sufficient: bool        # True = answer from RAG; False = fall back to web
    reasoning: str             # Brief explanation of the evaluation
    missing_info: list[str]    # What information is missing, if any


class AgentAnswer(BaseModel):
    """Final structured answer from the agent."""
    answer: str                # Natural language answer
    source: str                # 'internal_rag' | 'web_search' | 'combined'
    confidence: float          # 0.0 – 1.0
    citations: list[str]       # Source references
    game_title: Optional[str] = None


print("✅ Pydantic models defined")

## 3. Three Core Agent Tools

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOOL 1: retrieve_game
# Search the ChromaDB vector database for relevant game info
# ─────────────────────────────────────────────────────────────

def retrieve_game(query: str, n_results: int = 3) -> dict:
    """
    Tool 1: Retrieve game information from the internal vector database.
    
    Performs semantic search over the ChromaDB game collection.
    Returns the top matching documents with similarity scores.
    
    Args:
        query:     Natural language query about a game
        n_results: Number of results to return (default 3)
    
    Returns:
        dict with 'results' list and 'context' string for LLM
    """
    raw = collection.query(
        query_texts=[query],
        n_results=min(n_results, collection.count()),
        include=["documents", "metadatas", "distances"]
    )
    
    results = []
    for doc, meta, dist in zip(
        raw["documents"][0],
        raw["metadatas"][0],
        raw["distances"][0]
    ):
        results.append({
            "document":   doc,
            "metadata":   meta,
            "distance":   round(dist, 4),
            "similarity": round(1 - dist, 4),
        })
    
    # Build a compact context string for the LLM
    context_parts = []
    for r in results:
        context_parts.append(r["document"])
    context = "\n\n---\n\n".join(context_parts)
    
    return {
        "results": results,
        "context": context,
        "top_similarity": results[0]["similarity"] if results else 0.0
    }


print("✅ Tool 1: retrieve_game")

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOOL 2: evaluate_retrieval
# Assess whether RAG results are sufficient to answer the query
# ─────────────────────────────────────────────────────────────

def evaluate_retrieval(query: str, retrieval_context: str) -> RetrievalEvaluation:
    """
    Tool 2: Evaluate the quality of retrieved results.
    
    Uses an LLM to assess whether the retrieved context contains
    enough information to answer the query confidently.
    Returns a structured RetrievalEvaluation with confidence score.
    
    Args:
        query:             The original user query
        retrieval_context: The document text returned by retrieve_game
    
    Returns:
        RetrievalEvaluation with confidence, is_sufficient, reasoning, missing_info
    """
    system_prompt = """You are a retrieval quality evaluator for a video game research agent.
Your job is to assess whether the retrieved context is sufficient to answer a user query.

Respond ONLY with valid JSON matching this schema:
{
  "confidence": <float 0.0-1.0>,
  "is_sufficient": <true|false>,
  "reasoning": "<1-2 sentence explanation>",
  "missing_info": ["<what is missing, if anything>"]
}

Guidelines:
- confidence >= 0.75 AND is_sufficient=true: answer from internal knowledge
- confidence < 0.75 OR is_sufficient=false: fall back to web search
- If the exact game/company is found in context, set is_sufficient=true
- If context is about a different game entirely, set is_sufficient=false"""

    user_prompt = f"""User Query: {query}

Retrieved Context:
{retrieval_context[:2000]}  

Evaluate whether this context is sufficient to answer the query."""

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=300
    )
    
    raw = response.choices[0].message.content.strip()
    raw = raw.strip("```json").strip("```").strip()
    
    try:
        data = json.loads(raw)
        return RetrievalEvaluation(**data)
    except Exception:
        # Fallback evaluation based on response length heuristic
        return RetrievalEvaluation(
            confidence=0.4,
            is_sufficient=False,
            reasoning="Could not parse evaluation; defaulting to web search fallback.",
            missing_info=["Evaluation parsing failed"]
        )


print("✅ Tool 2: evaluate_retrieval")

In [ ]:
# ─────────────────────────────────────────────────────────────
# TOOL 3: game_web_search
# Fall back to Tavily web search when internal knowledge is insufficient
# ─────────────────────────────────────────────────────────────

def game_web_search(query: str, max_results: int = 3) -> dict:
    """
    Tool 3: Search the web for game information using Tavily API.
    
    Called when the internal RAG database does not have sufficient
    information to answer the user's query.
    
    Args:
        query:       The search query (should be specific to video games)
        max_results: Maximum number of web results to retrieve
    
    Returns:
        dict with 'results' list and 'context' string for LLM
    """
    # Add "video game" context to improve search relevance
    search_query = f"{query} video game"
    
    response = tavily_client.search(
        query=search_query,
        max_results=max_results,
        search_depth="basic",
        include_answer=True
    )
    
    results = []
    context_parts = []
    
    # Include Tavily's direct answer if available
    if response.get("answer"):
        context_parts.append(f"Direct Answer: {response['answer']}")
    
    for r in response.get("results", []):
        results.append({
            "title":   r.get("title", ""),
            "url":     r.get("url", ""),
            "content": r.get("content", "")[:500],  # truncate for context window
            "score":   r.get("score", 0.0),
        })
        context_parts.append(
            f"Source: {r.get('title', 'Unknown')} ({r.get('url', '')})"
            f"\n{r.get('content', '')[:400]}"
        )
    
    return {
        "results": results,
        "context": "\n\n".join(context_parts),
        "answer":  response.get("answer", "")
    }


print("✅ Tool 3: game_web_search")

## 4. State Machine

In [ ]:
class AgentState(Enum):
    """States in the UdaPlay agent workflow."""
    IDLE         = auto()   # Waiting for query
    RETRIEVING   = auto()   # Calling retrieve_game tool
    EVALUATING   = auto()   # Calling evaluate_retrieval tool
    WEB_SEARCH   = auto()   # Calling game_web_search tool
    GENERATING   = auto()   # Calling LLM to generate final answer
    REPORTING    = auto()   # Formatting and returning result
    ERROR        = auto()   # Error state


@dataclass
class AgentContext:
    """Holds all state for a single query turn."""
    query:              str
    state:              AgentState = AgentState.IDLE
    retrieval_result:   dict = field(default_factory=dict)
    evaluation:         Optional[RetrievalEvaluation] = None
    web_result:         dict = field(default_factory=dict)
    final_answer:       Optional[AgentAnswer] = None
    reasoning_trace:    list[str] = field(default_factory=list)
    used_web_search:    bool = False

    def log(self, message: str):
        self.reasoning_trace.append(f"[{self.state.name}] {message}")


print("✅ State machine defined")

## 5. Stateful Agent Class

In [ ]:
class UdaPlayAgent:
    """
    Stateful AI Research Agent for the Video Game Industry.
    
    Workflow (state machine):
        IDLE → RETRIEVING → EVALUATING → GENERATING → REPORTING
                                      ↘ WEB_SEARCH → GENERATING → REPORTING
    
    Maintains conversation history across multiple queries in a session.
    """
    
    CONFIDENCE_THRESHOLD = 0.75
    
    def __init__(self):
        self.conversation_history: list[dict] = []
        self.session_context: list[str] = []     # Accumulated facts from this session
        self.query_count: int = 0
        
        self.system_prompt = """You are UdaPlay, an expert AI research assistant specialising 
in the video game industry. You answer questions about game titles, developers, publishers, 
release dates, platforms, and genres.

Always:
- Cite your information sources
- Be specific and factual
- Use the provided context as your primary source
- Acknowledge uncertainty when present
- Reference previous conversation context when relevant"""
        
        self.conversation_history.append({
            "role": "system",
            "content": self.system_prompt
        })
        
        print("🎮 UdaPlay Agent initialised")
    
    # ── Public interface ──────────────────────────────────────────────
    
    def ask(self, query: str, verbose: bool = True) -> AgentAnswer:
        """Process a user query through the full agent workflow."""
        self.query_count += 1
        ctx = AgentContext(query=query)
        
        if verbose:
            print(f"\n{'='*65}")
            print(f"  Query #{self.query_count}: {query}")
            print(f"{'='*65}")
        
        try:
            self._transition(ctx, AgentState.RETRIEVING, verbose)
            self._do_retrieve(ctx, verbose)
            
            self._transition(ctx, AgentState.EVALUATING, verbose)
            self._do_evaluate(ctx, verbose)
            
            if not ctx.evaluation.is_sufficient or ctx.evaluation.confidence < self.CONFIDENCE_THRESHOLD:
                self._transition(ctx, AgentState.WEB_SEARCH, verbose)
                self._do_web_search(ctx, verbose)
            
            self._transition(ctx, AgentState.GENERATING, verbose)
            self._do_generate(ctx, verbose)
            
            self._transition(ctx, AgentState.REPORTING, verbose)
            self._do_report(ctx, verbose)
        
        except Exception as e:
            ctx.state = AgentState.ERROR
            ctx.log(f"Error: {e}")
            ctx.final_answer = AgentAnswer(
                answer=f"An error occurred processing your query: {e}",
                source="error",
                confidence=0.0,
                citations=[]
            )
        
        return ctx.final_answer
    
    def reset(self):
        """Reset conversation history (keep system prompt)."""
        self.conversation_history = [{"role": "system", "content": self.system_prompt}]
        self.session_context = []
        self.query_count = 0
        print("🔄 Agent conversation reset")
    
    # ── State machine transitions ─────────────────────────────────────
    
    def _transition(self, ctx: AgentContext, new_state: AgentState, verbose: bool):
        ctx.state = new_state
        if verbose:
            icons = {
                AgentState.RETRIEVING: "🔍",
                AgentState.EVALUATING: "⚖️",
                AgentState.WEB_SEARCH: "🌐",
                AgentState.GENERATING: "🤖",
                AgentState.REPORTING:  "📋",
                AgentState.ERROR:      "❌",
            }
            icon = icons.get(new_state, "➡️")
            print(f"\n  {icon} State: {new_state.name}")
    
    # ── Tool execution methods ────────────────────────────────────────
    
    def _do_retrieve(self, ctx: AgentContext, verbose: bool):
        ctx.log(f"Calling retrieve_game('{ctx.query}')")
        ctx.retrieval_result = retrieve_game(ctx.query)
        top_sim = ctx.retrieval_result.get("top_similarity", 0)
        n = len(ctx.retrieval_result.get("results", []))
        ctx.log(f"Retrieved {n} documents. Top similarity: {top_sim:.4f}")
        if verbose:
            print(f"     Retrieved {n} docs | Top similarity: {top_sim:.4f}")
            for r in ctx.retrieval_result.get("results", [])[:2]:
                print(f"     → {r['metadata']['title']} (sim={r['similarity']:.4f})")
    
    def _do_evaluate(self, ctx: AgentContext, verbose: bool):
        ctx.log("Calling evaluate_retrieval")
        ctx.evaluation = evaluate_retrieval(
            query=ctx.query,
            retrieval_context=ctx.retrieval_result.get("context", "")
        )
        ctx.log(
            f"Confidence: {ctx.evaluation.confidence:.2f} | "
            f"Sufficient: {ctx.evaluation.is_sufficient}"
        )
        if verbose:
            print(f"     Confidence: {ctx.evaluation.confidence:.2f} | "
                  f"Sufficient: {ctx.evaluation.is_sufficient}")
            print(f"     Reasoning: {ctx.evaluation.reasoning}")
            if ctx.evaluation.missing_info:
                print(f"     Missing: {ctx.evaluation.missing_info}")
    
    def _do_web_search(self, ctx: AgentContext, verbose: bool):
        ctx.used_web_search = True
        ctx.log(f"Calling game_web_search('{ctx.query}')")
        ctx.web_result = game_web_search(ctx.query)
        n = len(ctx.web_result.get("results", []))
        ctx.log(f"Web search returned {n} results")
        if verbose:
            print(f"     Web search returned {n} results")
            for r in ctx.web_result.get("results", [])[:2]:
                print(f"     → {r['title'][:60]}")
    
    def _do_generate(self, ctx: AgentContext, verbose: bool):
        ctx.log("Generating final answer with LLM")
        
        # Build context for the LLM
        rag_context  = ctx.retrieval_result.get("context", "")
        web_context  = ctx.web_result.get("context", "")
        
        context_block = "### Internal Database Results:\n" + (rag_context or "No results found.")
        if ctx.used_web_search and web_context:
            context_block += "\n\n### Web Search Results:\n" + web_context
        
        # Include conversation history context
        session_note = ""
        if self.session_context:
            session_note = "\n\nPrevious session context:\n" + "\n".join(self.session_context[-3:])
        
        user_msg = f"""Answer the following question about video games using the provided context.
Be specific, cite sources, and state confidence level.

Question: {ctx.query}
{session_note}

{context_block}

Respond in this JSON format:
{{
  "answer": "<natural language answer>",
  "source": "<internal_rag|web_search|combined>",
  "confidence": <0.0-1.0>,
  "citations": ["<source1>", "<source2>"],
  "game_title": "<primary game title if applicable, else null>"
}}"""
        
        # Add to conversation history for multi-turn context
        self.conversation_history.append({"role": "user", "content": user_msg})
        
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=self.conversation_history,
            temperature=0.2,
            max_tokens=600
        )
        
        assistant_reply = response.choices[0].message.content.strip()
        self.conversation_history.append({"role": "assistant", "content": assistant_reply})
        
        # Parse structured output
        raw = assistant_reply.strip("```json").strip("```").strip()
        try:
            data = json.loads(raw)
            ctx.final_answer = AgentAnswer(**data)
        except Exception:
            ctx.final_answer = AgentAnswer(
                answer=assistant_reply,
                source="combined" if ctx.used_web_search else "internal_rag",
                confidence=ctx.evaluation.confidence if ctx.evaluation else 0.5,
                citations=[]
            )
        
        # Save this answer to session context for future turns
        if ctx.final_answer and ctx.final_answer.game_title:
            self.session_context.append(
                f"Q: {ctx.query} → {ctx.final_answer.game_title}: "
                f"{ctx.final_answer.answer[:120]}"
            )
    
    def _do_report(self, ctx: AgentContext, verbose: bool):
        if verbose and ctx.final_answer:
            ans = ctx.final_answer
            print(f"\n  📝 ANSWER")
            print(f"  {'-'*61}")
            print(f"  {ans.answer}")
            print(f"  {'-'*61}")
            print(f"  Source     : {ans.source}")
            print(f"  Confidence : {ans.confidence:.2f}")
            if ans.citations:
                print(f"  Citations  : {' | '.join(ans.citations[:3])}")
            print(f"  Web Search : {'Yes' if ctx.used_web_search else 'No'}")
            print()


print("✅ UdaPlayAgent class defined")

## 6. Instantiate and Demonstrate the Agent

Running the agent on **5 example queries** as required by the rubric.

In [ ]:
agent = UdaPlayAgent()

In [ ]:
# ── Query 1: Developer lookup (should use internal RAG) ──────────────
answer1 = agent.ask("Who developed FIFA 21?")

In [ ]:
# ── Query 2: Release date (should use internal RAG) ──────────────────
answer2 = agent.ask("When was God of War Ragnarok released?")

In [ ]:
# ── Query 3: Platform lookup (internal RAG) ───────────────────────────
answer3 = agent.ask("What platform was Pokémon Red launched on?")

In [ ]:
# ── Query 4: Current/recent info (likely triggers web search) ─────────
answer4 = agent.ask("What is Rockstar Games working on right now?")

In [ ]:
# ── Query 5: Multi-turn — references previous context ─────────────────
answer5 = agent.ask("Who published the game you just told me about from Rockstar?")

## 7. Full Session Report

In [ ]:
print("\n" + "="*65)
print("  UDAPLAY SESSION REPORT")
print("="*65)

queries = [
    ("Who developed FIFA 21?",                          answer1),
    ("When was God of War Ragnarok released?",          answer2),
    ("What platform was Pokémon Red launched on?",      answer3),
    ("What is Rockstar Games working on right now?",    answer4),
    ("Who published the game from Rockstar?",           answer5),
]

for i, (q, a) in enumerate(queries, 1):
    print(f"\nQuery {i}: {q}")
    print(f"Answer : {a.answer}")
    print(f"Source : {a.source} | Confidence: {a.confidence:.2f}")
    if a.citations:
        print(f"Cited  : {', '.join(a.citations[:2])}")

print(f"\n{'='*65}")
print(f"  Total Queries : {agent.query_count}")
print(f"  Session Context Items Remembered: {len(agent.session_context)}")
print(f"{'='*65}")

## 8. Stand-Out Features

The following extras go beyond the base rubric:

In [ ]:
# ── Stand-Out 1: Structured JSON output alongside natural language ────
print("Structured JSON Output for Query 1:")
print(json.dumps(answer1.model_dump(), indent=2))

In [ ]:
# ── Stand-Out 2: Persistent long-term memory — session context ────────
print("Session Memory (facts accumulated this session):")
for i, fact in enumerate(agent.session_context, 1):
    print(f"  {i}. {fact[:120]}")

In [ ]:
# ── Stand-Out 3: Confidence threshold is configurable ─────────────────
print(f"Current confidence threshold: {UdaPlayAgent.CONFIDENCE_THRESHOLD}")
print("Adjust UdaPlayAgent.CONFIDENCE_THRESHOLD to make the agent")
print("more or less aggressive about falling back to web search.")

## 9. Summary

| Rubric Criterion | Implementation | Status |
|---|---|---|
| Tool: retrieve game from vector DB | `retrieve_game()` using ChromaDB semantic search | ✅ |
| Tool: evaluate retrieval quality | `evaluate_retrieval()` using LLM + Pydantic structured output | ✅ |
| Tool: web search fallback | `game_web_search()` using Tavily API | ✅ |
| Agent-first-attempts-internal-then-evaluates | State machine: RETRIEVE → EVALUATE → branch | ✅ |
| Stateful agent with conversation memory | `conversation_history` + `session_context` lists | ✅ |
| State machine / workflow abstraction | `AgentState` enum + `AgentContext` dataclass | ✅ |
| Clear structured answers with citations | `AgentAnswer` Pydantic model | ✅ |
| ≥3 example queries with reasoning trace | 5 queries demonstrated with full verbose output | ✅ |
| Structured JSON + natural language output | `answer.model_dump()` alongside prose | ✅ (stand-out) |
| Long-term session memory | `session_context` list persists across turns | ✅ (stand-out) |